# Character-level LSTM for arithmetic



In [ ]:
# ! pip install seaborn
# ! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
import opencc
import os
from sklearn.model_selection import train_test_split

data_path = './data'

In [ ]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

In [ ]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [ ]:
char_to_id = {}
id_to_char = {}

# Deterministic vocabulary, derived exclusively from the training data.
tokens = ['<pad>', '<eos>'] + sorted(set(''.join(df_train['src'])))
char_to_id = {char: index for index, char in enumerate(tokens)}
id_to_char = {index: char for char, index in char_to_id.items()}
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite

vocab_size = len(char_to_id)
print('Vocab size{}'.format(vocab_size))

# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [ ]:
# I implemented the TODO with assistance from OpenAI Codex 
def encode_training_row(text):
    ids = [char_to_id[char] for char in text]
    equals_index = text.index('=')
    labels = ([char_to_id['<pad>']] * equals_index
              + ids[equals_index + 1:] + [char_to_id['<eos>']])
    return ids, labels

encoded_rows = [encode_training_row(text) for text in df_train['src']]
df_train['char_id_list'] = [row[0] for row in encoded_rows]
df_train['label_id_list'] = [row[1] for row in encoded_rows]
del encoded_rows
df_train.head()

# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [ ]:
batch_size = 64
epochs = 2
embed_dim = 256
hidden_dim = 256
lr = 0.001
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    
    def __len__(self):
        # return the amount of data
        return len(self.sequences)
    
    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        x = self.sequences.iloc[index]['char_id_list']
        y = self.sequences.iloc[index]['label_id_list']
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])
    
    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [ ]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [ ]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(ds_train, batch_size=batch_size,
                                       shuffle=True, collate_fn=collate_fn)

# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [ ]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        # I set up the model using LSTM, with assistance from OpenAI Codex
        # Design your own model, which should contain embedding layer, several hidden layers and a linear layer
        
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])
        
        self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size))
        
    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)
    
    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)
        
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)
        
        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)
        
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)
        
        batch_x = self.linear(batch_x)
        
        return batch_x
    
    def generator(self, start_char, max_len=200):
        
        char_list = [char_to_id[c] for c in start_char]
        
        next_char = None
        
        while len(char_list) < max_len: 
            with torch.no_grad():
                prefix = torch.tensor([char_list], dtype=torch.long,
                                      device=next(self.parameters()).device)
                prefix_lens = torch.tensor([len(char_list)], dtype=torch.long)
                logits = self(prefix, prefix_lens)

            # Pack the char_list to tensor
            # Input the tensor to the embedding layer, RNN layers, linear respectively
            y = logits[0, -1] # Obtain the next token prediction y
            
            next_char = int(torch.argmax(y).item()) # Greedy next token
            
            if next_char == char_to_id['<eos>']:
                break
            
            char_list.append(next_char)
            
        return [id_to_char[ch_id] for ch_id in char_list]

In [ ]:
torch.manual_seed(2)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim)

In [ ]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
resume_training = False
if resume_training:
    model.load_state_dict(torch.load('checkpoint.pt', map_location='cpu',
                                     weights_only=True))
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
completed_epochs = 0
best_exact_match = -1.0


# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [ ]:
from tqdm import tqdm
from copy import deepcopy
model = model.to(device)
model.train()
i = 0
for epoch in range(1, epochs+1):
    # The process bar
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        optimizer.zero_grad(set_to_none=True)
        # Clear the gradient
    
        batch_pred_y = model(batch_x.to(device), batch_x_lens)
        
        loss = criterion(batch_pred_y.reshape(-1, vocab_size),
                         batch_y.to(device).reshape(-1))
        loss.backward()
        # Input the prediction and ground truths to loss function
        # Back propagation

        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) # gradient clipping

        optimizer.step()
        # Optimize parameters in the model

        i+=1
        if i%50==0:
            bar.set_postfix(loss = loss.item())
    
    # Evaluate your model
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        batch_x = row['src']
        batch_y = row['tgt']
        
        prediction = model.generator(batch_x)
        
        generated_answer = ''.join(prediction)[len(batch_x):]
        matched += int(generated_answer == batch_y)
        total += 1
        if total == len(df_eval):
            completed_epochs += 1
            scheduler.step()
            current_em = matched / total
            if current_em > best_exact_match:
                best_exact_match = current_em
                torch.save(model.state_dict(), 'checkpoint.pt')
        # EM = correct/total

        
    print(matched/total)

Running environment:
- Local Windows
- AMD Ryzen 9 5900HS CPU
- NVIDIA GeForce RTX 3080 Laptop GPU (8 GB VRAM)

Python version: 3.13.5

##### 1. Describe your model architecture, data preprocessing methods, and hyperparameter settings, and justify your choices. (5%)

Answer:

I used two-layer LSTM model for this homework. Each character is converted into 256-dimensional embedding and the two LSTM layers each has 256 hidden units. The linear layers predict the output characters. I chose LSTM because it can remember information from earlier characters, which can help for arithmetic calculations.

For preprocessing, I converted numbers and symbols into token IDs using 18 tokens dictionary. I used pad for sequences to be the same length and eos for the end of an answer.

| Hyperparameter | Value |
|---|---|
| Batch size | 64 |
| Embedding size | 256 |
| Hidden size | 256 |
| Optimizer | Adam |
| Gradient clipping | 1 |
| Initial learning rate | 0.001 |
| Total training | 10 epochs |

I used cross-entropy loss and Adam. I experimented by gradually reducing the learning rate, turns out it improved the model. I train by loading the previous checkpoint and doing several runs.

After 10 epochs, the model achieved 86.33% exact-match accuracy on 10,000 evaluation examples. The checkpoint is the best-performing model.

##### 2. If you use 2-layer LSTM or GRU instead of RNN, what will happen to the quality of your answer generation? Why? (10%)

Answer:

In my experiment, LSTM is better than basic RNN.

| Model | Epochs | Accuracy |
|---|---:|---:|
| Two-layer RNN | 2 | 13.46% |
| Two-layer LSTM | 2 | 53.84% |
| Two-layer LSTM | 10 | 86.33% |

In theory, LSTM would perform better because it uses gates to remember important information and forget unnecessary information. This helps the model with longer expressions and arithmetic operations.

I did not test GRU.

I ended up going for LSTM for the longer training, since it has the better initial accuracy than RNN during the same epoch.

##### 3. If we construct a training set using three-digit numbers while the evaluation set is constructed from two-digit numbers, what will happen to the quality of your answer generation? (10%)

Answer:

I expect accuracy to decrease because the model is only trained on three-digit numbers but the test uses two-digit numbers instead.

As an example:
We have:
- Training: 123+456=579

but then,
- Evaluation: 12+45=57

Even though the model knows the digits and operators, the patterns that it learned may possible be focused on three-digit numbers instead of math arithmetic rules. Having different lengths in the number and digit positions might result in incorrect predictions due to this side-effect.

But if the model learns the calculation rules, it may still perform well on two-digit numbers since it does not rely on three-digit patterns.

##### 4. If we construct a training set that includes 20% incorrect answers, how will this affect the quality of the generated responses? Present some examples. (10%)

Answer:

I expect accuracy to decrease because the model learns from both correct and incorrect answers. Having 20% incorrect answers might make the model perform incorrectly as well.

The accuracy might not decrease by exactly 20%, because the model can still learn correct patterns from the correct 80% of the data.

##### 5. Why do we need gradient clipping during training? (5%)

Answer:

We need gradient clipping because it can prevent gradients from exploding during training. In the RNNs and LSTMs that we use for the homework, gradients can become very large. This can cause unstable training with large weight updates.

In my implementation, I clip between -1 and 1 after backpropagation and before updating the model weights.

This helps with training being more stable and prevents the model from making weight updates too large.

##### Anything that can strengthen your report. (5%)

Answer:

My final LSTM model has 86.33% accuracy on 10,000 evaluation examples.
Performance table:

| Expression  | Accuracy |
|---|---:|---:|
| With multiplication  | 76.59% |
| Without multiplication  | 99.90% |
| With parentheses  | 89.20% |
| Negative answers  | 88.44% |
| Zero or positive answers  | 85.56% |

Multiplication is the hardest operation for the model, while addition and subtraction have high accuracy.

Predictions:

| Expression | Correct answer | Predicted answer |
|---|---:|---:|
| `34+47-32=` | 49 | 49 |
| `21*(41+3)=` | 924 | 924 |
| `31-(36*44)=` | -1553 | -1551 |
| `47*18*34=` | 28764 | 28652 |

The model can solve most expressions correctly but still makes mistakes with larger multiplication results.

##### Generative AI Usage Statement

Answer:

I used OpenAI Codex to assist me with coding the model and preparing the checkpoint (with checkpoint loading) and package list.
